In [19]:
import napari
import tifffile
import numpy as np
from pathlib import Path
from magicgui import magicgui
from magicgui.widgets import TextEdit

In [ ]:
def read_image_and_meta(path):
    """Read a tif as a (Z, C, Y, X) array along with the metadata needed to
    write it back out unchanged (pixel size / spacing / unit)."""
    with tifffile.TiffFile(path) as tif:
        series = max(tif.series, key=lambda s: s.size)
        arr = series.asarray()
        axes = series.axes
        ij_meta = dict(tif.imagej_metadata or {})

        def _rational(tag):
            if tag is None:
                return None
            val = tag.value
            if isinstance(val, tuple) and len(val) == 2:
                return val[0] / val[1] if val[1] else None
            return val

        xres = yres = resunit = None
        for page in tif.pages:
            if "XResolution" in page.tags:
                xres = _rational(page.tags.get("XResolution"))
                yres = _rational(page.tags.get("YResolution"))
                resunit_tag = page.tags.get("ResolutionUnit")
                resunit = int(resunit_tag.value) if resunit_tag is not None else None
                break

    meta = {
        "axes": axes,
        "imagej": ij_meta,
        "xres": xres,
        "yres": yres,
        "resunit": resunit,
    }
    return arr, meta



In [ ]:

def save_image(path, arr, meta):
    """Write `arr` to `path`, preserving the original pixel size / spacing /
    unit metadata.

    The channel / slice / frame counts are recomputed from `arr` (rather than
    copied from the source metadata) so the ImageJ header stays correct even
    when the channel count changes, e.g. when we add a 4th `valid` channel to a
    previously 3-channel stack.
    """
    ij = meta["imagej"]
    axes = meta["axes"]
    md = {"axes": axes}

    for key in ("spacing", "unit", "finterval", "fps", "mode"):
        if key in ij:
            md[key] = ij[key]

    for ax, name in (("Z", "slices"), ("C", "channels"), ("T", "frames")):
        if ax in axes:
            md[name] = arr.shape[axes.index(ax)]

    kwargs = {"imagej": True, "metadata": md}
    if meta["xres"] and meta["yres"]:
        kwargs["resolution"] = (meta["xres"], meta["yres"])
    if meta["resunit"] is not None:
        kwargs["resolutionunit"] = meta["resunit"]

    tifffile.imwrite(path, arr, **kwargs)


In [ ]:
class MaskCurator:
    def __init__(self, default_folder=None,
                 brightfield_channel=0, fluorescence_channel=1, mask_channel=2):
        self.default_folder = Path(default_folder) if default_folder else None
        self.brightfield_channel = brightfield_channel
        self.fluorescence_channel = fluorescence_channel
        self.mask_channel = mask_channel

        self.viewer = None
        self.path = None
        self.arr = None
        self.meta = None
        self.mask_layer = None
        self.log_widget = None
        self._orig_close = None

    def _log(self, message):
        """Append a message to the GUI log panel (falls back to print)."""
        if self.log_widget is not None:
            current = self.log_widget.value
            self.log_widget.value = (current + "\n" + message) if current else message
        else:
            print(message)

    def start(self):
        self.viewer = napari.Viewer(title="Mask curation")

        self.load_widget = magicgui(
            self._load_image,
            image_path={"label": "Image", "mode": "r",
                        "filter": "TIFF (*.tif *.tiff)"},
            call_button="Load image",
        )
        if self.default_folder and self.default_folder.exists():
            self.load_widget.image_path.value = self.default_folder

        self.save_widget = magicgui(self._save, call_button="Save")

        self.log_widget = TextEdit(value="", label="Log")
        try:
            self.log_widget.native.setReadOnly(True)
        except Exception:
            pass
        self.log_widget.min_height = 120
        self.log_widget.max_height = 300

        self.viewer.window.add_dock_widget(self.load_widget, area="right",
                                           name="Select image")
        self.viewer.window.add_dock_widget(self.save_widget, area="right",
                                           name="Save")
        self.viewer.window.add_dock_widget(self.log_widget, area="right",
                                           name="Log")

        # Also save the current image when the window is closed.
        qt_window = self.viewer.window._qt_window
        self._orig_close = qt_window.closeEvent
        qt_window.closeEvent = self._on_close

    def _load_image(self, image_path=Path()):
        image_path = Path(image_path)
        if not image_path.is_file():
            self._log("[WARN] Please select a tif file.")
            return

        try:
            arr, meta = read_image_and_meta(image_path)
        except Exception as e:
            self._log(f"[ERROR] Could not read {image_path.name}: "
                      f"{type(e).__name__}: {e}")
            return

        self.path = image_path
        self.arr, self.meta = arr, meta

        brightfield = self.arr[:, self.brightfield_channel, :, :]
        fluorescence = self.arr[:, self.fluorescence_channel, :, :]
        mask = self.arr[:, self.mask_channel, :, :]

        # Replace any layers from a previously loaded image.
        self.viewer.layers.clear()
        self.viewer.add_image(brightfield, name="brightfield",
                              colormap="gray", blending="additive", scale=(5, 2, 2))
        self.viewer.add_image(fluorescence, name="fluorescence",
                              colormap="green", blending="additive", scale=(5, 2, 2))
        self.mask_layer = self.viewer.add_labels(mask.astype(np.int32),
                                                 name="mask", scale=(5, 2, 2))
        self.viewer.layers.selection.active = self.mask_layer
        self.viewer.title = self.path.name
        self._log(f"[OK] Loaded {self.path.name}")

    def _save(self):
        if self.mask_layer is None or self.path is None:
            self._log("[WARN] Load an image before saving.")
            return

        self.arr[:, self.mask_channel, :, :] = self.mask_layer.data.astype(self.arr.dtype)
        save_image(self.path, self.arr, self.meta)
        self._log(f"[OK] Saved -> {self.path.name}")

    def _on_close(self, event):
        if self.mask_layer is not None and self.path is not None:
            self._save()
        self._orig_close(event)

In [ ]:
# Default folder the file explorer opens in (you can change to wherever your images live, or just navigate to the folder each time from the GUI).
masks_folder = Path(r"Z:\Bel\IMAGES_FOR_VASCUMAP_RETRAINING\fluorescent_cells_tifs\3_channel_images_to_curate")

In [ ]:
curator = MaskCurator(default_folder=masks_folder)
curator.start()